# 3.01 — Baseline: DummyClassifier
**Etapa 1 · Tech Challenge POSTECH MLOps**

Estabelece o **piso absoluto** de performance usando a estratégia `most_frequent`.

### O que este notebook faz
1. Carrega os splits `train.parquet` / `test.parquet` de `data/processed/`
2. Roda validação cruzada estratificada 5-fold
3. Salva figuras em `reports/figures/baselines/` ← **commitado no git**
4. Registra dataset, parâmetros, métricas e artefatos no MLflow ← **ignorado no git**

### Rastreamento MLflow
Backend **SQLite** (`mlflow.db`) configurado em `churn_telecom/config.py`:
```python
MLFLOW_TRACKING_URI = f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
MLFLOW_ARTIFACT_URI = (PROJECT_ROOT / "mlartifacts").as_uri()
```
Ambos ignorados pelo `.gitignore` — regenerados ao executar este notebook.  
As figuras em `reports/figures/baselines/` permanecem disponíveis sem executar.


## 0. Imports e Setup

In [ ]:
"""Imports, seed global e configuração de logging via config.py."""
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# ── Resolve raiz do projeto (notebooks/ está um nível abaixo) ─────────────────
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Configuração central — única fonte de verdade ─────────────────────────────
from churn_telecom.config import (
    DATA_PROCESSED,
    MLFLOW_EXPERIMENT,
    MODELS_DIR,
    PROJECT_ROOT,
    RANDOM_STATE,
    REPORTS_FIGURES,
    TARGET,
    LABEL_COL,
    TARGET_SNAKE,
    get_logger,
    log_dataset_to_mlflow,
    mlflow_run,
)

# ── Visualização ──────────────────────────────────────────────────────────────
from churn_telecom.plots import save_confusion_matrix, save_roc_curve

# ── MLflow ────────────────────────────────────────────────────────────────────
import mlflow
import mlflow.sklearn

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

import matplotlib
matplotlib.use("Agg")

# ── Seed global ───────────────────────────────────────────────────────────────
np.random.seed(RANDOM_STATE)

# ── Logger estruturado — sem print() ─────────────────────────────────────────
logger = get_logger("3.01_baseline_dummy")
logger.info("ROOT            : %s", ROOT)
logger.info("PROJECT_ROOT    : %s", PROJECT_ROOT)
logger.info("RANDOM_STATE    : %d", RANDOM_STATE)
logger.info("MLflow version  : %s", mlflow.__version__)

warnings.filterwarnings("ignore", category=UserWarning)


## 1. Paths

Todos os paths são derivados de `config.py` — sem strings hardcoded.


In [ ]:
# ── Inputs ────────────────────────────────────────────────────────────────────
TRAIN_PATH        = DATA_PROCESSED / "train.parquet"
TEST_PATH         = DATA_PROCESSED / "test.parquet"
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.pkl"

# ── Outputs: figuras commitadas no git ────────────────────────────────────────
# reports/figures/baselines/ existe independente do MLflow
FIGURES_DIR = REPORTS_FIGURES / "baselines"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Validação de pré-requisitos ───────────────────────────────────────────────
_required = [TRAIN_PATH, TEST_PATH, PREPROCESSOR_PATH]
_missing  = [p for p in _required if not p.exists()]

if _missing:
    for p in _missing:
        logger.error("FALTANDO: %s", p.relative_to(PROJECT_ROOT))
    raise FileNotFoundError(
        "Execute os notebooks 1.01 → 1.03 antes deste. "
        f"Faltando: {[str(p.relative_to(PROJECT_ROOT)) for p in _missing]}"
    )

for p in _required:
    logger.info("OK  %s", p.relative_to(PROJECT_ROOT))


## 2. Carregamento dos Dados

Os splits já estão estratificados e fixados com `random_state=42` (notebook `1.03`).  
O target é `TARGET = "Churn Value"` conforme definido em `config.py`.


In [ ]:
# ── Carrega dados já transformados pelo 1.03 ──────────────────────────────────
# train.parquet = pós-preprocessor + pós-SMOTE+ENN  (5395 × 31)
# test.parquet  = pós-preprocessor, original          (1405 × 31)
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

TARGET_COL = "churn_value"   # nome snake_case — definido no 1.03

X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL].astype(int)
X_test  = test.drop(columns=[TARGET_COL])
y_test  = test[TARGET_COL].astype(int)

logger.info("Train: %d × %d | Churn=%.2f%%", *X_train.shape, y_train.mean()*100)
logger.info("Test : %d × %d | Churn=%.2f%%", *X_test.shape,  y_test.mean()*100)

## 3. Pipeline — DummyClassifier

`DummyClassifier(strategy='most_frequent')`:
- Prediz **sempre** a classe majoritária (`Churn Value = 0`)
- Não lê nenhuma feature — serve como piso absoluto
- Revela o perigo de usar acurácia: com ~73.5% de negativos, atinge ~73.5% sem aprender nada

> `SelectKBest` **não é aplicado** aqui — o Dummy ignora features por definição.


In [ ]:
# Parâmetros do DummyClassifier — declarados explicitamente para logging no MLflow
DUMMY_PARAMS = {
    "strategy":     "most_frequent",
    "random_state": RANDOM_STATE,
}

logger.info("DUMMY_PARAMS: %s", DUMMY_PARAMS)

In [ ]:
# Pipeline simples: dados já estão transformados
dummy_pipeline = Pipeline([
    ("classifier", DummyClassifier(**DUMMY_PARAMS)),
])

## 4. Validação Cruzada Estratificada (5-fold)

> **Mapeamento sklearn → chave no MLflow**
>
> | Conceito | `scoring=` (sklearn) | Chave logada no MLflow |
> |---|---|---|
> | AUC-ROC | `"roc_auc"` | `cv_roc_auc_mean` |
> | PR-AUC | `"average_precision"` | `cv_average_precision_mean` |
> | F1 | `"f1"` | `cv_f1_mean` |
> | Recall | `"recall"` | `cv_recall_mean` |
> | Precision | `"precision"` | `cv_precision_mean` |
> | MCC | `"matthews_corrcoef"` | `cv_matthews_corrcoef_mean` |
>
> `cv_pr_auc_mean` **não existe** — o sklearn chama PR-AUC de `average_precision`.


In [ ]:
SCORING = [
    "roc_auc",
    "average_precision",   # PR-AUC — nome canônico do sklearn
    "f1",
    "recall",
    "precision",
    "matthews_corrcoef",
]

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logger.info(
    "cross_validate | strategy=%s | folds=%d | stratified=True",
    DUMMY_PARAMS["strategy"], 5,
)

cv_results = cross_validate(
    dummy_pipeline,
    X_train, y_train,
    cv=CV,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

# Sumário CV
cv_summary: dict[str, dict[str, float]] = {}
for key, values in cv_results.items():
    if key.startswith("test_"):
        name = key.removeprefix("test_")
        cv_summary[name] = {
            "mean": float(values.mean()),
            "std":  float(values.std()),
        }
        logger.info(
            "  cv %-26s  mean=%+.4f  std=%.4f",
            name, values.mean(), values.std(),
        )


## 5. Avaliação no Hold-out

In [ ]:
dummy_pipeline.fit(X_train, y_train)

y_pred  = dummy_pipeline.predict(X_test)
y_proba = dummy_pipeline.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Dicionário com TODAS as chaves que serão logadas no MLflow
# Nomes espelham exatamente os gerados pelo sklearn em cross_validate
hold_out: dict[str, float] = {
    "test_roc_auc":           float(roc_auc_score(y_test, y_proba)),
    "test_average_precision": float(average_precision_score(y_test, y_proba)),
    "test_f1":                float(f1_score(y_test, y_pred, zero_division=0)),
    "test_recall":            float(recall_score(y_test, y_pred, zero_division=0)),
    "test_precision":         float(precision_score(y_test, y_pred, zero_division=0)),
    "test_mcc":               float(matthews_corrcoef(y_test, y_pred)),
    "test_npv":               float(tn / (tn + fn)) if (tn + fn) > 0 else 0.0,
    "test_accuracy":          float((tp + tn) / len(y_test)),
    # células da matriz — úteis para cálculo de custo de negócio
    "test_tp": float(tp), "test_tn": float(tn),
    "test_fp": float(fp), "test_fn": float(fn),
}

for k, v in hold_out.items():
    logger.info("  hold-out %-30s  %.4f", k, v)


## 6. Figuras

As figuras são salvas em `reports/figures/baselines/` **antes** de serem logadas
no MLflow. Isso garante que estejam disponíveis no repositório mesmo sem executar
o MLflow localmente.

```
reports/figures/baselines/     ← commitado no git
    dummy_confusion_matrix.png
    dummy_roc_curve.png

mlartifacts/<exp_id>/<run_id>/  ← ignorado no git (cópia dos mesmos arquivos)
```


In [ ]:
# ── 6.1 Confusion Matrix ──────────────────────────────────────────────────────
# Usa plots.py — interface padronizada do projeto
cm_path = save_confusion_matrix(
    y_true = y_test,
    y_pred = y_pred,
    path   = FIGURES_DIR / "dummy_confusion_matrix.png",
    title  = "Confusion Matrix — DummyClassifier (most_frequent)",
)

# ── 6.2 ROC Curve ─────────────────────────────────────────────────────────────
roc_path = save_roc_curve(
    y_true      = y_test,
    y_proba     = y_proba,
    path        = FIGURES_DIR / "dummy_roc_curve.png",
    model_name  = "DummyClassifier",
    title       = f"ROC Curve — DummyClassifier  (AUC = {hold_out['test_roc_auc']:.3f})",
)

logger.info("Figuras salvas em: %s", FIGURES_DIR.relative_to(PROJECT_ROOT))


## 7. Registro no MLflow

Usa o context manager `mlflow_run()` de `config.py`, que internamente chama
`setup_mlflow()` (configura URI + experimento) e inicia o run.

```
mlflow.db          ← SQLite local, ignorado no git
mlartifacts/       ← artefatos binários, ignorados no git
```

O que é registrado:
1. **Dataset** — via `log_dataset_to_mlflow()` com hash MD5 do parquet
2. **Parâmetros** — hiperparâmetros do modelo e contexto do experimento
3. **Métricas CV** — `cv_<scorer>_mean` / `_std` para cada scorer
4. **Métricas hold-out** — `test_<scorer>` no conjunto de teste
5. **Artefatos** — cópias das figuras já salvas em `reports/figures/`
6. **Modelo** — pipeline serializado via `mlflow.sklearn.log_model`


In [ ]:
with mlflow_run(run_name="dummy_most_frequent") as run:
    RUN_ID = run.info.run_id
    EXP_ID = run.info.experiment_id
    logger.info("Run iniciado | run_id=%s | exp_id=%s", RUN_ID, EXP_ID)

    # ── 7.1 Dataset ───────────────────────────────────────────────────────────
    # log_dataset_to_mlflow usa mlflow.log_input + hash MD5 do parquet
    # Garante rastreabilidade: qual versão dos dados gerou este run
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test,  y_test,  split="test",  source_path=TEST_PATH)
    logger.info("Datasets logados com hash MD5")

    # ── 7.2 Parâmetros ────────────────────────────────────────────────────────
    mlflow.log_params({
        # Modelo
        "model_type":           "DummyClassifier",
        **DUMMY_PARAMS,
        "feature_selection":    "none",   # Dummy ignora features
        # Validação
        "cv_folds":             5,
        "cv_stratified":        True,
        "cv_scoring":           str(SCORING),
        # Dados
        "preprocessor_path":    str(PREPROCESSOR_PATH.relative_to(PROJECT_ROOT)),
        "dataset_train":        str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
        "dataset_test":         str(TEST_PATH.relative_to(PROJECT_ROOT)),
        "n_train":              int(len(X_train)),
        "n_test":               int(len(X_test)),
        "n_features":           int(X_train.shape[1]),
        "churn_rate_train":     round(float(y_train.mean()), 4),
        "churn_rate_test":      round(float(y_test.mean()), 4),
        "target_col":           TARGET,
    })
    logger.info("Parâmetros logados")

    # ── 7.3 Métricas CV ───────────────────────────────────────────────────────
    # Padrão: cv_<scorer>_mean  e  cv_<scorer>_std
    # Exemplo: cv_average_precision_mean  (NÃO cv_pr_auc_mean)
    for scorer_name, stats in cv_summary.items():
        mlflow.log_metric(f"cv_{scorer_name}_mean", stats["mean"])
        mlflow.log_metric(f"cv_{scorer_name}_std",  stats["std"])
    logger.info("Métricas CV logadas (%d scorers × 2)", len(cv_summary))

    # ── 7.4 Métricas hold-out ─────────────────────────────────────────────────
    for metric_name, metric_value in hold_out.items():
        mlflow.log_metric(metric_name, metric_value)
    logger.info("Métricas hold-out logadas (%d)", len(hold_out))

    # ── 7.5 Artefatos ─────────────────────────────────────────────────────────
    # As figuras já existem em reports/figures/baselines/ (commitadas no git).
    # O log_artifact cria uma cópia em mlartifacts/ (ignorada no git).
    # Estrutura gerada:
    #   mlartifacts/<exp_id>/<run_id>/figures/dummy_confusion_matrix.png
    #   mlartifacts/<exp_id>/<run_id>/figures/dummy_roc_curve.png
    for fig_path in [cm_path, roc_path]:
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
        logger.info(
            "Artefato logado: %s → mlartifacts/.../figures/",
            fig_path.name,
        )

    # ── 7.6 Modelo ────────────────────────────────────────────────────────────
    mlflow.sklearn.log_model(
        sk_model       = dummy_pipeline,
        artifact_path  = "dummy_classifier",
        registered_model_name = None,   # sem registry no baseline
    )

    logger.info(
        "Run finalizado | experiment=%s | run_id=%s",
        MLFLOW_EXPERIMENT, RUN_ID,
    )
    logger.info(
        "Resumo | roc_auc=%.4f | avg_precision=%.4f | f1=%.4f | recall=%.4f",
        hold_out["test_roc_auc"],
        hold_out["test_average_precision"],
        hold_out["test_f1"],
        hold_out["test_recall"],
    )


## 8. Sumário

In [ ]:
"""Tabela de resultados — referência para comparação com o notebook 3.02."""
summary = pd.DataFrame([{
    "Modelo":      "DummyClassifier (most_frequent)",
    "AUC-ROC":     hold_out["test_roc_auc"],
    "PR-AUC":      hold_out["test_average_precision"],
    "F1":          hold_out["test_f1"],
    "Recall":      hold_out["test_recall"],
    "Precision":   hold_out["test_precision"],
    "MCC":         hold_out["test_mcc"],
    "NPV":         hold_out["test_npv"],
    "Accuracy":    hold_out["test_accuracy"],
}]).set_index("Modelo").round(4)

logger.info("\n%s", summary.T.to_string())
summary.T


### Interpretação

| Chave MLflow | Valor | Significado |
|---|---|---|
| `test_roc_auc` | 0.500 | Equivalente ao acaso |
| `test_average_precision` | ≈ 0.265 | ≈ proporção de churners (piso da PR-AUC) |
| `test_f1` | 0.000 | Nenhum churner detectado |
| `test_recall` | 0.000 | 100% dos FN — perda máxima de negócio |
| `test_accuracy` | ≈ 0.735 | ⚠️ Enganosa — reflete apenas a classe dominante |

### Localização dos artefatos

| Artefato | Path | Git |
|---|---|---|
| Confusion matrix | `reports/figures/baselines/dummy_confusion_matrix.png` | ✅ commitado |
| ROC curve | `reports/figures/baselines/dummy_roc_curve.png` | ✅ commitado |
| MLflow run | `mlflow.db` + `mlartifacts/` | ❌ ignorado (regenerar: `make baselines`) |
| Log | `logs/3.01_baseline_dummy.log` | ❌ ignorado |

**Próximo**: `3.02_vab_baseline_logistic.ipynb` — Regressão Logística + `SelectKBest`.
